# Experiment 001 — 5×5 m local majority, 100-frame vote
## Идея / Idea
Для каждого кадра считаем игроков в квадрате 5×5 м вокруг наблюдаемого мяча. Команда с большим числом игроков получает **один** голос. При равенстве (включая 0:0) и отсутствии мяча голоса нет.
Команду определяет большинство голосов за последние **100 кадров, включая текущий** (4 секунды). Равенство сохраняет последний ответ; если его ещё нет, выдаётся `X`.
Это детерминированная эвристика, без обучения и без подбора параметров по публичной разметке. Гомография и видео не используются.

Остановки — отдельная эвристика по выходу мяча за границу, неподвижности и движению игроков. Smart Tagging только уменьшает время подтверждения неподвижного мяча при стандартном положении; сам по себе тег не означает `D` и не задаёт владельца.
Все прогнозы, метрики, графики и логи остаются в `local_review/experiments/001_box5_vote100/`. В Git публикуются только исходный ноутбук, конфигурация и инструкция воспроизведения.

In [ ]:
from pathlib import Path
from collections import deque
import sys, json, time, warnings, subprocess, importlib.metadata
import numpy as np

# Locate this experiment without personal absolute paths.
# Находим эксперимент без личных абсолютных путей.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "games.py").exists())
EXP = ROOT / "sol/001_box5_vote100"
OUT = ROOT / "local_review/experiments/001_box5_vote100"
OUT.mkdir(parents=True, exist_ok=True)
# Keep generated diagnostics untracked even in a fresh clone.
# Исключаем локальные результаты из Git и при запуске в новом клоне.
local_ignore = ROOT / "local_review/.gitignore"
if not local_ignore.exists():
    local_ignore.write_text("*\n")
CFG = json.loads((EXP / "config.json").read_text())
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "evaluation"))
from games import GAME, GAMES, asset_paths, half_video_blocks
from starter.baseline import team_of_each_track
from starter.load_smart_tagging import load as load_smart_tagging
import possession_metrics as pm

N = GAMES[GAME]["n_video_frames"]
FPS = GAMES[GAME]["fps"]
PITCH = np.array(GAMES[GAME]["pitch"], dtype=np.float32)
BLOCKS = half_video_blocks(GAME)
PATHS = asset_paths()
started = time.perf_counter()
# All thresholds are fixed before any ground truth is loaded.
# Все пороги зафиксированы до чтения эталонной разметки.
print(json.dumps(CFG, indent=2))


In [ ]:
# Load only ATD positions; raw speed units are inconsistent in the source documentation.
# Читаем только координаты ATD: единицы готовой скорости неоднозначны.
track_teams = team_of_each_track(PATHS["atd_meta"])
players = np.full((N, len(track_teams), 2), np.nan, dtype=np.float32)
ball = np.full((N, 2), np.nan, dtype=np.float32)
read_count = 0
with open(PATHS["atd_raw"]) as handle:
    for i, line in enumerate(handle):
        frame_id, player_text, ball_text = line.strip().split(":")
        assert int(frame_id) == i + 1, "ATD/video alignment mismatch"
        player_values = np.fromstring(player_text.replace(";", ","), sep=",").reshape(-1, 3)
        assert len(player_values) == len(track_teams)
        players[i] = player_values[:, :2] * PITCH
        ball[i] = np.fromstring(ball_text, sep=",")[:2] * PITCH
        read_count += 1
assert read_count == N
ball_seen = np.isfinite(ball).all(axis=1)
players_seen = np.isfinite(players).all(axis=2)

# The window is an axis-aligned SQUARE, not a radius-2.5 m circle.
# Окно — КВАДРАТ вдоль осей поля, не круг радиусом 2.5 м.
def box_votes(player_xy, ball_xy, team_ids, width_m):
    inside = (np.isfinite(player_xy).all(axis=2)
              & np.isfinite(ball_xy).all(axis=1)[:, None]
              & (np.abs(player_xy - ball_xy[:, None, :]) <= width_m / 2).all(axis=2))
    home_count = (inside & (team_ids == 0)).sum(axis=1)
    away_count = (inside & (team_ids == 1)).sum(axis=1)
    # +1 means H, -1 means A, 0 means abstention for this frame.
    # +1 — H, -1 — A, 0 — нет голоса в этом кадре.
    votes = np.sign(home_count - away_count).astype(np.int8)
    return home_count, away_count, votes

home_count, away_count, frame_votes = box_votes(players, ball, track_teams, CFG["box_width_m"])
# A narrow synthetic check verifies corners, ties, and missing-ball handling.
# Проверяем на искусственных точках углы квадрата, ничью и отсутствие мяча.
test_players = np.array([[[2.5, 2.5], [2.6, 0]], [[0, 0], [0, 0]], [[0, 0], [0, 0]]])
test_ball = np.array([[0., 0.], [0., 0.], [np.nan, np.nan]])
assert box_votes(test_players, test_ball, np.array([0, 1]), 5)[2].tolist() == [1, 0, 0]

# A set-piece interval provides context only; its team tag is not used as a possession label.
# Интервал стандарта — только контекст; его Team не копируем в метку владения.
set_piece_context = np.zeros(N, dtype=bool)
for event in load_smart_tagging():
    if (event["code"] or "").strip().upper() == "SET PIECES":
        start = max(0, event["start_frame"])
        stop = min(N, event["end_frame"])
        set_piece_context[start:stop] = True
print("ATD and automatic set-piece context loaded; no ground truth accessed.")


## Движение и остановки / Motion and stoppages
Скорость рассчитывается из изменения координат за 10 кадров (0,4 с), а не из неоднозначного канала `s`. Для игроков берётся медиана по минимум 10 доступным трекам; скачки более 12 м/с исключаются как ненадёжные.
Для мяча требуем непрерывную наблюдаемость между двумя точками, чтобы не измерять движение через пропуск.

Вход в `D`: мяч больше чем на 0,5 м за границей 3 кадра подряд; **или** низкая скорость игроков 25 кадров и неподвижный мяч 25 кадров (15 в контексте стандарта); **или** низкая скорость игроков 25 кадров и мяч отсутствует 50 кадров.
Выход из `D`: мяч внутри поля движется быстрее 0,8 м/с 3 кадра; либо медианное движение игроков быстрее 1,8 м/с 12 кадров. Это предположения первой версии, а не официальное определение остановки.
При начале тайма и каждой остановке очищаем историю голосов. Будущие координаты не используются; Smart Tagging уже является предоставленной офлайн-разметкой.

In [ ]:
lag = CFG["motion_lag_frames"]
dt = lag / FPS
player_motion = np.full(N, np.nan)
ball_motion = np.full(N, np.nan)

# Calculate motion separately in each half, so halftime never creates a false displacement.
# Движение считаем отдельно внутри таймов, не соединяя их через перерыв.
for _, first, last in BLOCKS:
    stop = last + 1
    speed = np.linalg.norm(players[first + lag:stop] - players[first:stop - lag], axis=2) / dt
    speed[(speed > CFG["max_player_motion_mps"]) | ~np.isfinite(speed)] = np.nan
    enough = np.isfinite(speed).sum(axis=1) >= CFG["minimum_motion_tracks"]
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="All-NaN slice encountered", category=RuntimeWarning)
        medians = np.nanmedian(speed, axis=1)
    medians[~enough] = np.nan
    player_motion[first + lag:stop] = medians
    seen = ball_seen[first:stop]
    cumulative = np.r_[0, np.cumsum(seen)]
    continuous = (cumulative[lag + 1:] - cumulative[:-(lag + 1)]) == lag + 1
    ball_speed = np.linalg.norm(ball[first + lag:stop] - ball[first:stop - lag], axis=1) / dt
    ball_speed[~continuous] = np.nan
    ball_motion[first + lag:stop] = ball_speed

outside = ball_seen & ((ball < -CFG["outside_margin_m"]) | (ball > PITCH + CFG["outside_margin_m"])).any(axis=1)
inside_field = ball_seen & (ball >= 0).all(axis=1) & (ball <= PITCH).all(axis=1)

def detect_dead_ball():
    dead = np.zeros(N, dtype=bool)
    # Diagnostic counts record triggers, not ground-truth stopping reasons.
    # Счётчики показывают срабатывания эвристик, а не истинные причины остановок.
    reasons = {"outside": 0, "stationary": 0, "missing_and_slow": 0}
    for _, first, last in BLOCKS:
        state = False
        outside_run = missing_run = stationary_run = slow_run = restart_ball_run = restart_players_run = 0
        for i in range(first, last + 1):
            outside_run = outside_run + 1 if outside[i] else 0
            missing_run = missing_run + 1 if not ball_seen[i] else 0
            stationary_run = stationary_run + 1 if ball_motion[i] <= CFG["stationary_ball_mps"] else 0
            slow_run = slow_run + 1 if player_motion[i] <= CFG["slow_players_mps"] else 0
            restart_ball_run = restart_ball_run + 1 if inside_field[i] and ball_motion[i] >= CFG["restart_ball_mps"] else 0
            restart_players_run = restart_players_run + 1 if player_motion[i] >= CFG["restart_players_mps"] else 0
            if not state:
                stationary_threshold = (CFG["set_piece_stationary_confirm_frames"] if set_piece_context[i]
                                        else CFG["stationary_ball_confirm_frames"])
                slow = slow_run >= CFG["slow_players_confirm_frames"]
                if outside_run >= CFG["outside_confirm_frames"]:
                    state = True
                    reasons["outside"] += 1
                elif slow and stationary_run >= stationary_threshold:
                    state = True
                    reasons["stationary"] += 1
                elif slow and missing_run >= CFG["missing_ball_confirm_frames"]:
                    state = True
                    reasons["missing_and_slow"] += 1
            elif not outside[i] and (restart_ball_run >= CFG["restart_ball_confirm_frames"]
                                    or restart_players_run >= CFG["restart_players_confirm_frames"]):
                state = False
                # Reset evidence so a previous stoppage cannot immediately trigger a new one.
                # Сбрасываем накопленные признаки предыдущей остановки.
                outside_run = missing_run = stationary_run = slow_run = 0
            dead[i] = state
    return dead, reasons

dead, dead_trigger_counts = detect_dead_ball()

def vote_labels(votes, dead_mask, blocks, window):
    labels = np.full(len(votes), "X", dtype="<U1")
    for _, first, last in blocks:
        history = deque()
        total = 0
        previous = "X"
        for i in range(first, last + 1):
            if dead_mask[i]:
                labels[i] = "D"
                history.clear()
                total = 0
                previous = "X"
                continue
            # Zero votes still consume a time slot: this is 100 FRAMES, not 100 observed votes.
            # Нулевой голос занимает место: окно из 100 КАДРОВ, не из 100 непустых голосов.
            current = int(votes[i])
            history.append(current)
            total += current
            if len(history) > window:
                total -= history.popleft()
            if total > 0:
                previous = "H"
            elif total < 0:
                previous = "A"
            labels[i] = previous
    return labels

# Verify exact window length, tie retention, half resets, and clearing at D.
# Проверяем длину окна, сохранение ответа при ничьей и сброс при остановке.
assert vote_labels(np.array([1, 0, -1]), np.zeros(3, bool), [(1, 0, 2)], 2).tolist() == ["H", "H", "A"]
assert vote_labels(np.array([1, 0, 0]), np.array([False, True, False]), [(1, 0, 2)], 100).tolist() == ["H", "D", "X"]
assert vote_labels(np.array([1, 0, 0]), np.zeros(3, bool), [(1, 0, 0), (2, 2, 2)], 100).tolist() == ["H", "X", "X"]
labels = vote_labels(frame_votes, dead, BLOCKS, CFG["vote_window_frames"])
SUBMISSION = OUT / "box5_vote100.csv"
with SUBMISSION.open("w") as handle:
    handle.write("frame,label\n")
    handle.writelines(f"{i},{label}\n" for i, label in enumerate(labels))
print("Predictions generated before loading any ground truth.")


## Проверка и оценка / Validation and evaluation
Эталон читается только после сохранения полного файла прогнозов. Оценка — только на публичных 5 минутах.
Сравниваем с авторскими baseline без изменения параметров. Все отчёты и статистика сохраняются локально, отдельно от папки решения.

In [ ]:
validation = subprocess.run([sys.executable, str(ROOT / "evaluation/validate_submission.py"), str(SUBMISSION)],
                            capture_output=True, text=True)
(OUT / "validation.log").write_text(validation.stdout + validation.stderr)
assert validation.returncode == 0, "Submission validation failed; see local log"

truth = pm.label_codes(pm.read_label_csv(str(ROOT / "evaluation/gt/BAR-ATH_possession_gt_public.csv")))
result = pm.evaluate(truth, pm.label_codes(labels), bootstrap_n=CFG["bootstrap_draws"])
(OUT / "metrics.json").write_text(json.dumps(result, indent=2))

comparison = []
for name, prediction in [(CFG["experiment"], labels)]:
    comparison.append({"name": name, "macroF1": result["macro_f1"], "spellF1": result["spell_f1_mean"],
                       "possessions": result["n_pred_spells"], "D_precision": result["per_class"]["D"]["precision"],
                       "D_recall": result["per_class"]["D"]["recall"], "D_F1": result["per_class"]["D"]["f1"]})
for baseline in sorted((ROOT / "evaluation/results").glob("baseline_*.csv")):
    reference = pm.evaluate(truth, pm.label_codes(pm.read_label_csv(str(baseline))))
    comparison.append({"name": baseline.stem, "macroF1": reference["macro_f1"], "spellF1": reference["spell_f1_mean"],
                       "possessions": reference["n_pred_spells"], "D_precision": reference["per_class"]["D"]["precision"],
                       "D_recall": reference["per_class"]["D"]["recall"], "D_F1": reference["per_class"]["D"]["f1"]})
(OUT / "comparison.json").write_text(json.dumps(comparison, indent=2))

graded = np.zeros(N, dtype=bool)
for _, first, last in BLOCKS:
    graded[first:last + 1] = True
public = truth != pm.X
facts = {"elapsed_s": time.perf_counter() - started,
         "dead_trigger_counts_full_match": dead_trigger_counts,
         "graded_frame_votes": {"H": int(((frame_votes == 1) & graded).sum()),
                                "A": int(((frame_votes == -1) & graded).sum()),
                                "none": int(((frame_votes == 0) & graded).sum())},
         "public_frame_votes": {"H": int(((frame_votes == 1) & public).sum()),
                                "A": int(((frame_votes == -1) & public).sum()),
                                "none": int(((frame_votes == 0) & public).sum())},
         "public_no_vote_ball_missing": int((public & ~ball_seen).sum()),
         "public_no_vote_seen_equal_counts": int((public & ball_seen & (frame_votes == 0)).sum()),
         "public_prediction_counts": {label: int(((labels == label) & public).sum()) for label in ["H", "A", "D", "X"]},
         "environment": {name: importlib.metadata.version(name) for name in ["numpy", "pandas", "nbclient", "nbformat", "ipykernel"]}}
(OUT / "diagnostics.json").write_text(json.dumps(facts, indent=2))
print(json.dumps(comparison, indent=2))
print("Public D precision/recall and all diagnostics saved locally.")
